# Microproyecto 3
# Finetuning de Llama 3.2 1B para preguntas de selección múltiple

**Curso:** Modelos avanzados para el Procesamiento de Lenguaje Natural  
**Dataset:** RACE  
**Modelo base:** `meta-llama/Llama-3.2-1B`  
**Técnica de ajuste:** LoRA  
**Entorno esperado:** Google Colab con GPU

---

## Índice

- [0. Setup de Colab, rutas y dependencias](#sec0)
- [1. Librerías e hiperparámetros](#sec1)
- [2. Carga y preparación del dataset RACE](#sec2)
- [3. Construcción del prompt y de la respuesta objetivo](#sec3)
- [4. Evaluación por probabilidad de las opciones](#sec4)
- [5. Evaluación del modelo original](#sec5)
- [6. Preparación de datos para finetuning](#sec6)
- [7. Finetuning con LoRA](#sec7)
- [8. Evaluación del modelo ajustado](#sec8)
- [9. Análisis comparativo, conclusiones y guardado](#sec9)
- [10. Carga del checkpoint para verificación](#sec10)



<a id="sec0"></a>
# 0. Setup de entorno, rutas y dependencias

El microproyecto está pensado para ejecutarse preferiblemente en **Google Colab con GPU**. Antes de correrlo en Colab, se recomienda activar:

`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.

La primera celda detecta automáticamente si el notebook está corriendo en Colab. Si no está en Colab, usa una carpeta local y evita intentar montar Google Drive.

El modelo `meta-llama/Llama-3.2-1B` requiere acceso aceptado en Hugging Face.


In [ ]:
import os
import sys
import subprocess

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB = IN_COLAB

if IN_COLAB:
    drive.mount('/content/drive')

packages = [
    'pandas==2.2.2',
    'transformers>=4.46.0,<5.0.0',
    'datasets',
    'accelerate',
    'peft',
    'bitsandbytes',
    'huggingface_hub',
    'matplotlib',
    'tqdm',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

print('Ejecutando en Colab:', IN_COLAB)
print('Dependencias instaladas o verificadas.')


In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import login, HfFolder
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    get_peft_model_state_dict,
    PeftModel,
)

print('pandas:', pd.__version__)


<a id="sec1"></a>
# 1. Librerías e hiperparámetros

Se centralizan las rutas, tamaños de subconjuntos, hiperparámetros de LoRA y parámetros de entrenamiento. Para la entrega final se recomienda dejar `N_TEST = None`, de modo que la evaluación se realice sobre todo el split `test` filtrado.


In [ ]:
seed = 99
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if COLAB:
    DRIVE_BASE = '/content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Microproyecto_3'
else:
    DRIVE_BASE = './Microproyecto_3'

RUN_NAME    = 'microproyecto_3'
DATA_DIR    = f'{DRIVE_BASE}/Data'
OUTPUT_DIR  = f'{DRIVE_BASE}/outputs'
ADAPTER_DIR = f'{OUTPUT_DIR}/llama32_1b_race_lora_adapter'
CKPT_FINAL  = f'{OUTPUT_DIR}/llama32_1b_race_lora_checkpoint.pt'
RESULTS_CSV = f'{OUTPUT_DIR}/resultados_microproyecto_3.csv'
CONCLUSIONS_MD = f'{OUTPUT_DIR}/conclusiones_microproyecto_3.md'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = 'meta-llama/Llama-3.2-1B'
DATASET_NAME = 'race'
DATASET_CONFIG = 'all'

MAX_ARTICLE_CHARS = 800

N_TRAIN = 3000
N_VAL   = 300
N_TEST  = None

MAX_SEQ_LEN = 1024
MAX_ANSWER_TOKENS = 96

NORMALIZE_BY_LENGTH = False

USE_4BIT = True

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
MAX_STEPS = 600
EVAL_STEPS = 100
LOGGING_STEPS = 25
SAVE_STEPS = 200

print('Drive base:', DRIVE_BASE)
print('Output dir:', OUTPUT_DIR)
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
assert torch.cuda.is_available(), (
    'Este notebook está diseñado para Colab con GPU. '
    'Activa Runtime > Change runtime type > GPU antes de continuar.'
)

def get_compute_dtype():
    """Define el tipo de dato más conveniente según la GPU disponible."""
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

compute_dtype = get_compute_dtype()
print('compute_dtype:', compute_dtype)


## 1.1 Token de Hugging Face

El token se usa de forma directa para autenticar la sesión de Colab y acceder al modelo `meta-llama/Llama-3.2-1B`, que es un repositorio con acceso restringido. Esta autenticación evita depender de `notebook_login()` o de los secretos de Colab.

In [ ]:
import os
from huggingface_hub import login, HfFolder

HF_TOKEN = 'hf_TU_TOKEN_AQUI'

if not HF_TOKEN or HF_TOKEN == 'PEGA_AQUI_TU_TOKEN':
    raise ValueError('Debe definir un token válido de Hugging Face.')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_TOKEN'] = HF_TOKEN

login(token=HF_TOKEN, add_to_git_credential=False)
HfFolder.save_token(HF_TOKEN)

print('Token de Hugging Face configurado.')


<a id="sec2"></a>
# 2. Carga y preparación del dataset RACE

El dataset RACE contiene un artículo, una pregunta, cuatro opciones y una respuesta correcta. El enunciado pide filtrar tanto entrenamiento como evaluación usando únicamente ejemplos con `len(article) < 800`.


In [ ]:
raw_ds = load_dataset(DATASET_NAME, DATASET_CONFIG)
# raw_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, trust_remote_code=True)

raw_ds


In [ ]:
LETTERS = ['A', 'B', 'C', 'D']
LETTER_TO_IDX = {letter: i for i, letter in enumerate(LETTERS)}

def answer_to_index(answer):
    """Convierte la respuesta del dataset a índice 0..3."""
    if isinstance(answer, int):
        return int(answer)

    answer = str(answer).strip().upper()
    if answer in LETTER_TO_IDX:
        return LETTER_TO_IDX[answer]
    if answer.isdigit():
        return int(answer)
    raise ValueError(f'Respuesta no reconocida: {answer}')

def valid_example(x):
    """Filtro principal del enunciado y validaciones mínimas de formato."""
    try:
        ans_idx = answer_to_index(x['answer'])
    except Exception:
        return False

    return (
        len(x['article']) < MAX_ARTICLE_CHARS
        and isinstance(x['options'], list)
        and len(x['options']) == 4
        and 0 <= ans_idx < 4
    )

filtered_ds = raw_ds.filter(valid_example)

for split in filtered_ds:
    print(f'{split:10s}: {len(filtered_ds[split]):,} ejemplos después del filtro')


In [ ]:
def select_subset(ds, n=None, seed=seed):
    """Selecciona un subconjunto reproducible o retorna todo el split si n=None."""
    ds = ds.shuffle(seed=seed)
    if n is None:
        return ds
    return ds.select(range(min(n, len(ds))))

train_ds = select_subset(filtered_ds['train'],      N_TRAIN)
val_ds   = select_subset(filtered_ds['validation'], N_VAL)
test_ds  = select_subset(filtered_ds['test'],       N_TEST)

print('Train:', len(train_ds))
print('Val:  ', len(val_ds))
print('Test: ', len(test_ds))


In [ ]:
example = train_ds[0]
print('ARTICLE:\n', example['article'][:600])
print('\nQUESTION:\n', example['question'])
print('\nOPTIONS:')
for letter, option in zip(LETTERS, example['options']):
    print(f'{letter}. {option}')
print('\nANSWER:', example['answer'], '→', example['options'][answer_to_index(example['answer'])])


<a id="sec3"></a>
# 3. Construcción del prompt y de la respuesta objetivo

El prompt se mantiene corto para reducir consumo de memoria. La continuación esperada durante entrenamiento es el texto de la respuesta correcta.


In [ ]:
PROMPT_VERSION = 'option_text_short'

def clean_text(text):
    return ' '.join(str(text).replace('\n', ' ').split())

def build_prompt(example):
    article = clean_text(example['article'])
    question = clean_text(example['question'])
    options = [clean_text(o) for o in example['options']]
    options_text = '\n'.join([f'{letter}. {option}' for letter, option in zip(LETTERS, options)])

    prompt = (
        f'Passage:\n{article}\n\n'
        f'Question:\n{question}\n\n'
        f'Options:\n{options_text}\n\n'
        'Answer:'
    )
    return prompt

def correct_option_text(example):
    idx = answer_to_index(example['answer'])
    return clean_text(example['options'][idx])

def build_training_answer(example, tokenizer):
    return ' ' + correct_option_text(example) + tokenizer.eos_token

print(build_prompt(example))
print('\nRespuesta objetivo:', repr(' ' + correct_option_text(example)))


<a id="sec4"></a>
# 4. Evaluación por probabilidad de las opciones

Para cada ejemplo se calcula la suma de log-probabilidades de los tokens de cada opción dada la pregunta y el artículo:

\[
\hat{s}=\arg\max_{s \in S} \log P(s|c)
\]

La respuesta predicha es la opción con mayor puntaje.


In [ ]:
def load_tokenizer(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    return tokenizer

def load_base_model(model_name=MODEL_NAME, use_4bit=USE_4BIT):
    """Carga Llama 3.2 1B en modo cuantizado o en precisión media."""
    kwargs = {
        'device_map': 'auto',
        'dtype': compute_dtype,
    }

    if use_4bit:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, token=HF_TOKEN, **kwargs)
    except TypeError:
        # Compatibilidad con versiones antiguas de transformers que aún no aceptan dtype.
        kwargs['torch_dtype'] = kwargs.pop('dtype')
        model = AutoModelForCausalLM.from_pretrained(model_name, token=HF_TOKEN, **kwargs)

    model.config.use_cache = False
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return model

tokenizer = load_tokenizer()
print('Vocab size:', len(tokenizer))
print('pad_token:', tokenizer.pad_token, tokenizer.pad_token_id)
print('eos_token:', tokenizer.eos_token, tokenizer.eos_token_id)


In [ ]:
def model_device(model):
    return next(model.parameters()).device

@torch.no_grad()
def score_options(model, tokenizer, prompt, options, normalize_by_length=False):
    model.eval()
    device = model_device(model)

    max_prompt_tokens = MAX_SEQ_LEN - MAX_ANSWER_TOKENS
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=max_prompt_tokens,
    )['input_ids']

    rows_input_ids = []
    rows_labels = []
    answer_lengths = []

    for option in options:
        answer_ids = tokenizer(
            ' ' + clean_text(option),
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_ANSWER_TOKENS,
        )['input_ids']

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids

        rows_input_ids.append(input_ids)
        rows_labels.append(labels)
        answer_lengths.append(max(1, len(answer_ids)))

    max_len = max(len(x) for x in rows_input_ids)
    pad_id = tokenizer.pad_token_id

    batch_input_ids = []
    batch_labels = []
    batch_attention = []

    for input_ids, labels in zip(rows_input_ids, rows_labels):
        pad_len = max_len - len(input_ids)
        batch_input_ids.append(input_ids + [pad_id] * pad_len)
        batch_labels.append(labels + [-100] * pad_len)
        batch_attention.append([1] * len(input_ids) + [0] * pad_len)

    input_ids = torch.tensor(batch_input_ids, dtype=torch.long, device=device)
    labels = torch.tensor(batch_labels, dtype=torch.long, device=device)
    attention_mask = torch.tensor(batch_attention, dtype=torch.long, device=device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    mask = shift_labels.ne(-100)

    safe_labels = shift_labels.masked_fill(~mask, 0)
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)

    scores = (token_log_probs * mask).sum(dim=1)

    if normalize_by_length:
        lengths = torch.tensor(answer_lengths, dtype=scores.dtype, device=scores.device)
        scores = scores / lengths

    return scores.detach().to(torch.float32).cpu().numpy().tolist()

def predict_example(model, tokenizer, example, normalize_by_length=NORMALIZE_BY_LENGTH):
    prompt = build_prompt(example)
    options = [clean_text(o) for o in example['options']]
    scores = score_options(model, tokenizer, prompt, options, normalize_by_length)
    pred_idx = int(np.argmax(scores))
    true_idx = answer_to_index(example['answer'])

    return {
        'pred_idx': pred_idx,
        'true_idx': true_idx,
        'pred_letter': LETTERS[pred_idx],
        'true_letter': LETTERS[true_idx],
        'pred_text': options[pred_idx],
        'true_text': options[true_idx],
        'scores': scores,
        'correct': pred_idx == true_idx,
    }

def evaluate_accuracy(model, tokenizer, dataset, max_examples=None, desc='Evaluando'):
    if max_examples is not None:
        dataset = dataset.select(range(min(max_examples, len(dataset))))

    rows = []
    correct = 0

    for i, ex in enumerate(tqdm(dataset, desc=desc)):
        pred = predict_example(model, tokenizer, ex)
        correct += int(pred['correct'])

        rows.append({
            'i': i,
            'correct': pred['correct'],
            'true_letter': pred['true_letter'],
            'pred_letter': pred['pred_letter'],
            'true_text': pred['true_text'],
            'pred_text': pred['pred_text'],
            'scores': pred['scores'],
        })

    accuracy = correct / len(dataset) if len(dataset) else 0.0
    return accuracy, pd.DataFrame(rows)


<a id="sec5"></a>
# 5. Evaluación del modelo original

Primero se evalúa `Llama-3.2-1B` sin finetuning. Esta será la línea base para comparar contra el modelo ajustado con LoRA.


In [ ]:
base_model = load_base_model()
base_model.eval()

acc_original, pred_original = evaluate_accuracy(
    base_model,
    tokenizer,
    test_ds,
    max_examples=None,
    desc='Evaluando modelo original'
)

print(f'Accuracy modelo original: {acc_original:.4f}')
display(pred_original.head())


In [ ]:
# Errores de ejemplo del modelo original
errors_original = pred_original[pred_original['correct'] == False].head(5)
display(errors_original[['i', 'true_letter', 'pred_letter', 'true_text', 'pred_text']])


<a id="sec6"></a>
# 6. Preparación de datos para finetuning

Para el entrenamiento, los tokens del contexto reciben label `-100`. Así la función de costo ignora el prompt y calcula pérdida únicamente sobre los tokens de la respuesta.


In [ ]:
def tokenize_for_training(example):
    """Tokeniza prompt + respuesta y enmascara el contexto con labels=-100."""
    prompt = build_prompt(example)
    answer = build_training_answer(example, tokenizer)

    max_prompt_tokens = MAX_SEQ_LEN - MAX_ANSWER_TOKENS

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=max_prompt_tokens,
    )['input_ids']

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_ANSWER_TOKENS,
    )['input_ids']

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    attention_mask = [1] * len(input_ids)

    # Padding fijo para que el default_data_collator pueda trabajar sin collator propio.
    pad_len = MAX_SEQ_LEN - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len
        attention_mask += [0] * pad_len
    else:
        input_ids = input_ids[:MAX_SEQ_LEN]
        labels = labels[:MAX_SEQ_LEN]
        attention_mask = attention_mask[:MAX_SEQ_LEN]

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }

train_tok = train_ds.map(tokenize_for_training, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_for_training, remove_columns=val_ds.column_names)

train_tok.set_format(type='torch')
val_tok.set_format(type='torch')

print(train_tok)
print(val_tok)


In [ ]:
sample_tok = train_tok[0]
labels = sample_tok['labels']
input_ids = sample_tok['input_ids']

masked_context_tokens = int((labels == -100).sum())
trainable_answer_tokens = int((labels != -100).sum())

print('input_ids:', input_ids.shape)
print('labels:', labels.shape)
print('Tokens ignorados con -100:', masked_context_tokens)
print('Tokens usados para pérdida:', trainable_answer_tokens)

answer_token_ids = labels[labels != -100].tolist()
print('\nRespuesta decodificada desde labels:')
print(tokenizer.decode(answer_token_ids))

assert trainable_answer_tokens > 0, 'No hay tokens de respuesta para entrenar.'
assert labels[0].item() == -100, 'Los tokens iniciales del contexto deben ignorarse con -100.'


<a id="sec7"></a>
# 7. Finetuning con LoRA

Se usa LoRA para entrenar una fracción pequeña de parámetros del modelo. Los argumentos de entrenamiento restauran automáticamente el checkpoint con menor pérdida de validación mediante `load_best_model_at_end=True`.


In [ ]:
if USE_4BIT:
    try:
        base_model = prepare_model_for_kbit_training(
            base_model,
            gradient_checkpointing_kwargs={'use_reentrant': False}
        )
    except TypeError:
        base_model = prepare_model_for_kbit_training(base_model)
        try:
            base_model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={'use_reentrant': False}
            )
        except TypeError:
            base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable / total if total else 0
    return trainable, total, pct

trainable_params, total_params, trainable_pct = count_trainable_parameters(model)
print(f'Parámetros entrenables: {trainable_params:,}')
print(f'Parámetros totales: {total_params:,}')
print(f'Porcentaje entrenable: {trainable_pct:.4f}%')


In [ ]:
common_args = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=EVAL_STEPS,
    save_total_limit=3,
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    remove_unused_columns=False,
    optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

try:
    training_args = TrainingArguments(eval_strategy='steps', save_strategy='steps', **common_args)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy='steps', save_strategy='steps', **common_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=default_data_collator,
)


In [ ]:
train_result = trainer.train()
train_result

print(f'Mejor checkpoint: {trainer.state.best_model_checkpoint}')
print(f'Mejor val_loss:   {trainer.state.best_metric:.4f}')


In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
train_logs = logs[logs['loss'].notna()][['step', 'loss']]
eval_logs = logs[logs['eval_loss'].notna()][['step', 'eval_loss']]

display(train_logs.tail())
display(eval_logs.tail())

plt.figure(figsize=(8, 4))
if len(train_logs) > 0:
    plt.plot(train_logs['step'], train_logs['loss'], label='Train loss')
if len(eval_logs) > 0:
    plt.plot(eval_logs['step'], eval_logs['eval_loss'], label='Val loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Curvas de entrenamiento — LoRA')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

base_model_config_path = Path(ADAPTER_DIR) / 'base_model_config.json'
model.config.to_json_file(str(base_model_config_path))

adapter_state = get_peft_model_state_dict(model)
adapter_state_cpu = {k: v.detach().cpu() for k, v in adapter_state.items()}

checkpoint = {
    'base_model': MODEL_NAME,
    'prompt_version': PROMPT_VERSION,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'max_seq_len': MAX_SEQ_LEN,
    'max_answer_tokens': MAX_ANSWER_TOKENS,
    'normalize_by_length': NORMALIZE_BY_LENGTH,
    'lora_config': lora_config.to_dict(),
    'adapter_state_dict': adapter_state_cpu,
    'base_model_config_path': str(base_model_config_path),
}

torch.save(checkpoint, CKPT_FINAL)
print(f'Adapter guardado en: {ADAPTER_DIR}')
print(f'Config base guardada en: {base_model_config_path}')
print(f'Checkpoint torch.save guardado en: {CKPT_FINAL}')


<a id="sec8"></a>
# 8. Evaluación del modelo ajustado

Se evalúa el modelo con LoRA sobre el mismo conjunto `test` usado en la línea base. La comparación debe hacerse con el mismo criterio: probabilidad de cada opción dada la pregunta y el artículo.


In [ ]:
model.eval()

acc_finetuned, pred_finetuned = evaluate_accuracy(
    model,
    tokenizer,
    test_ds,
    max_examples=None,
    desc='Evaluando modelo finetuned'
)

print(f'Accuracy modelo finetuned: {acc_finetuned:.4f}')
display(pred_finetuned.head())


In [ ]:
errors_finetuned = pred_finetuned[pred_finetuned['correct'] == False].head(5)
display(errors_finetuned[['i', 'true_letter', 'pred_letter', 'true_text', 'pred_text']])


<a id="sec9"></a>
# 9. Análisis comparativo y guardado de entregables

La rúbrica pide comparar el modelo original contra el modelo ajustado. En esta sección se consolida el resultado en una tabla y se guarda un CSV de respaldo.


In [ ]:
results = pd.DataFrame([
    {
        'modelo': 'Llama-3.2-1B original',
        'split': 'test filtrado len(article) < 800',
        'n_eval': len(test_ds),
        'accuracy': acc_original,
    },
    {
        'modelo': 'Llama-3.2-1B + LoRA',
        'split': 'test filtrado len(article) < 800',
        'n_eval': len(test_ds),
        'accuracy': acc_finetuned,
    },
])

results['accuracy_pct'] = (results['accuracy'] * 100).round(2)

improvement = acc_finetuned - acc_original
improvement_pct_points = improvement * 100

display(results)
print(f'Mejora absoluta: {improvement_pct_points:.2f} puntos porcentuales')
print(f'Mejora relativa aproximada: {(improvement / acc_original * 100) if acc_original > 0 else float("nan"):.2f}%')

results.to_csv(RESULTS_CSV, index=False)
print(f'Resultados guardados en: {RESULTS_CSV}')


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results['modelo'], results['accuracy_pct'])
plt.ylabel('Accuracy (%)')
plt.title('Comparación de desempeño en RACE')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.show()


## 9.1 Conclusiones

La siguiente celda genera el texto de conclusiones usando los valores obtenidos durante la ejecución.


In [ ]:
def safe_best_eval_summary(eval_logs):
    if eval_logs is None or len(eval_logs) == 0:
        return None
    best_row = eval_logs.loc[eval_logs['eval_loss'].idxmin()]
    return int(best_row['step']), float(best_row['eval_loss'])

best_eval = safe_best_eval_summary(eval_logs if 'eval_logs' in globals() else None)
correct_original = int(round(acc_original * len(test_ds)))
correct_finetuned = int(round(acc_finetuned * len(test_ds)))

best_ckpt_path = getattr(trainer.state, 'best_model_checkpoint', None)
best_ckpt_metric = getattr(trainer.state, 'best_metric', None)

if best_eval is None:
    best_eval_text = 'No se registró pérdida de validación en los logs del entrenamiento.'
else:
    best_step, best_loss = best_eval
    best_eval_text = (
        f'La mejor pérdida de validación se observó en el paso {best_step}, '
        f'con un valor de {best_loss:.4f}. El entrenamiento restauró ese checkpoint '
        'antes de realizar la evaluación final.'
    )

conclusions_text = f'''# Conclusiones del microproyecto 3

En este microproyecto se ajustó el modelo `Llama-3.2-1B` para resolver una tarea de preguntas de selección múltiple sobre el dataset RACE. Antes del entrenamiento se aplicó el filtro solicitado, conservando únicamente ejemplos con artículos de menos de {MAX_ARTICLE_CHARS} caracteres. La evaluación final se realizó sobre {len(test_ds)} ejemplos del conjunto `test` filtrado.

El modelo original obtuvo un accuracy de {acc_original:.4f}, equivalente a {acc_original*100:.2f}%, con aproximadamente {correct_original} respuestas correctas. Después del finetuning con LoRA, el modelo ajustado obtuvo un accuracy de {acc_finetuned:.4f}, equivalente a {acc_finetuned*100:.2f}%, con aproximadamente {correct_finetuned} respuestas correctas. La mejora absoluta fue de {(acc_finetuned - acc_original)*100:.2f} puntos porcentuales.

La comparación se hizo usando el mismo criterio en ambos casos: seleccionar la opción con mayor log-probabilidad condicionada al artículo y la pregunta. Esto permite evaluar el modelo como modelo de lenguaje, sin convertir la tarea en una clasificación tradicional.

LoRA permitió realizar el ajuste de manera eficiente, entrenando {trainable_params:,} parámetros, equivalentes aproximadamente al {trainable_pct:.4f}% del total de parámetros cargados. Esto es adecuado para un entorno como Colab, donde la memoria de GPU es limitada.

{best_eval_text}

En conclusión, el finetuning con LoRA mejoró el desempeño de `Llama-3.2-1B` en la tarea RACE. Aunque el modelo todavía puede fallar en preguntas que requieren inferencia más fina o comprensión detallada del texto, los resultados muestran que el ajuste con un subconjunto limitado de datos fue suficiente para mejorar el rendimiento frente al modelo base.
'''.strip()

print(conclusions_text)

Path(CONCLUSIONS_MD).write_text(conclusions_text, encoding='utf-8')
print(f'\nConclusiones guardadas en: {CONCLUSIONS_MD}')


In [ ]:
checkpoint['metrics'] = {
    'accuracy_original': float(acc_original),
    'accuracy_finetuned': float(acc_finetuned),
    'improvement_pct_points': float((acc_finetuned - acc_original) * 100),
    'n_eval': int(len(test_ds)),
}
checkpoint['trainable_parameters'] = {
    'trainable_params': int(trainable_params),
    'total_params': int(total_params),
    'trainable_pct': float(trainable_pct),
}
if 'conclusions_text' in globals():
    checkpoint['conclusions'] = conclusions_text

torch.save(checkpoint, CKPT_FINAL)
print(f'Checkpoint actualizado con métricas: {CKPT_FINAL}')


<a id="sec10"></a>
# 10. Carga del checkpoint para verificación

Esta sección permite verificar que el adapter entrenado se puede volver a cargar para evaluación. Es útil correrla en una sesión nueva de Colab antes de entregar.


In [ ]:

RUN_RELOAD_TEST = False

if RUN_RELOAD_TEST:
    reloaded_base = load_base_model()
    reloaded_model = PeftModel.from_pretrained(reloaded_base, ADAPTER_DIR)
    reloaded_model.eval()

    acc_reloaded, _ = evaluate_accuracy(
        reloaded_model,
        tokenizer,
        test_ds,
        max_examples=50,
        desc='Verificación rápida checkpoint'
    )
    print(f'Accuracy reload test en 50 ejemplos: {acc_reloaded:.4f}')
